In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from numpy.linalg import eig
import pandas as pd

#import matplotlib  
#matplotlib.use('Agg')  # Use a non-GUI backend

In [2]:
#cat='C1'
#path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'
#path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/zbox/z20.0/'
#path=f'C:/shehani/postdoc_work/ML_Diffusion/md_data/c4_nmeth_nh2o_box1/47h2o_3obnh/'
path = f'D:/ML_Diffusion/dual_cat/c2c6_94h2o_z17.5/'
#df = pd.DataFrame(columns=['nrep', '#hops' 'Dnonhop_mean', 'Dnonhop_se', 'D_hop', 'Dtot'])

nrep=4
cat='c2c6'
oh=2
nsteps=5001


In [3]:
def oh_xyz(nrep, nsteps):
    #nrep = number of replicas
    #nsteps = number of steps in fs/MD_freq (ex: for 10 ps simulation with MD_freq=10, nsteps=10,000/10=1000)
    #path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'
    
    #extracting x,y, z cordinates from oh_id.dat files
    x_oh = np.zeros((nrep,nsteps))
    y_oh = np.zeros((nrep,nsteps))
    z_oh = np.zeros((nrep,nsteps))
    indx_oh= np.zeros((nrep,nsteps))

    for i in range(nrep):
        #with open(path+ f'oh_id_{i+1}.dat', 'r') as oh_id:
        #with open(path+ f'oh1.dat', 'r') as oh_id:
        with open(path+ f'oh{oh}.{i+1}_id.dat', 'r') as oh_id:
            xyz= oh_id.readlines()[:nsteps]
            
            for j in range(len(xyz)):   
                indx_oh[i,j]=int(xyz[j].split()[1])
                x_oh[i,j]=float(xyz[j].split()[2])
                y_oh[i,j]=float(xyz[j].split()[3])
                z_oh[i,j]=float(xyz[j].split()[4])
                
    #dtime=  np.arange(0.01, (0.01*ndt+0.01), 0.01)
    return(x_oh, y_oh, z_oh, indx_oh)




In [4]:
x_oh, y_oh, z_oh, indx_oh= oh_xyz( nrep, nsteps)

In [5]:
#path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/zbox/z20.0/'
d_rep = pd.DataFrame(columns=['nrep', '#hops', 'D_nonhop (mean)', 'D_nonhop (SE)', 'D_hop', 'D_tot'])
d_rep = d_rep.astype(float)
for kk in range(1,nrep+1):
    dtime = np.zeros(nsteps)
    D1x=np.zeros(kk)
    D1y=np.zeros(kk)
    D1z=np.zeros(kk)
    D1=np.zeros(kk)
    rx=[]
    ry=[]
    rz=[]
    t=[]
    ax=[]
    ay=[]
    az=[]
    t_hop=[]

    nhop=0
    for ll in range(kk):
        int_indx= indx_oh[ll,0]
        int_x= x_oh[ll,0]
        int_y= y_oh[ll,0]
        int_z= z_oh[ll,0]
        int_t=0
        hop=0
        for jj in range(nsteps):
            dtime[jj]= np.round(jj*0.01, 2)
            
            if indx_oh[ll,jj]==int_indx:
                pass
                #print('nonhop',ll, jj, indx_oh[ll,jj],dtime[jj], x_oh[ll,jj])
            else:
                hop=hop+1
                #t_hop.append(dtime[jj])
                rx.append((x_oh[ll,jj-1]-int_x)**2)
                ry.append((y_oh[ll,jj-1]-int_y)**2)
                rz.append((z_oh[ll,jj-1]-int_z)**2)
                t.append(dtime[jj-1]-int_t)
                #print('hop',ll, jj, indx_oh[ll,jj])
                #print(int_t, int_indx, int_x, int_y)
                #print(dtime[jj-1], indx_oh[ll,jj-1], x_oh[ll,jj-1], y_oh[ll,jj-1])     
                #print(t[-1], rx[-1], ry[-1])   
                ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2)
                ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2)
                az.append((z_oh[ll,jj]-z_oh[ll,jj-1])**2)
                #print(jj)
    
                if hop>1:
                    #ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2)
                    #ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2)
                    #print(ll,jj,(x_oh[ll,jj]-x_oh[ll,jj-1])**2 )
                    #print(x_oh[ll,jj],x_oh[ll,jj-1])
                    #print(dtime[jj-1],int_t)
                    t_hop.append(dtime[jj-1]-int_t)
    
                
                int_indx= indx_oh[ll,jj]
                int_x= x_oh[ll,jj]
                int_y= y_oh[ll,jj]
                int_z= z_oh[ll,jj]
                int_t=dtime[jj]
                #print(hop,int_t)
    
                #print(indx_oh[ll,jj], x_oh[ll,jj], x_oh[ll,jj-1], dtime[jj],int_t)
        nhop= nhop+hop      
        #print(ll,hop)
        rx.append((x_oh[ll,-1]-int_x)**2)
        ry.append((y_oh[ll,-1]-int_y)**2)
        rz.append((z_oh[ll,-1]-int_z)**2)
        t.append(dtime[-1]-int_t)
        
        D1x[ll]=(np.mean(rx))/(2*np.mean(t))
        D1y[ll]=(np.mean(ry))/(2*np.mean(t))
        D1z[ll]=(np.mean(rz))/(2*np.mean(t))
        D1[ll]=(np.mean(rx)+np.mean(ry)+np.mean(rz))/(6*np.mean(t))
    
    dmean= np.mean(D1)
    dstd= np.std(D1)
    std_err= dstd/np.sqrt(kk)
    
    D2x=(np.mean(ax))/(2*np.mean(t_hop))
    D2y=(np.mean(ay))/(2*np.mean(t_hop))
    D2z=(np.mean(az))/(2*np.mean(t_hop))
    D2=(np.mean(ax)+np.mean(ay)+np.mean(az))/(6*np.mean(t_hop))
    D=dmean+D2

    d_rep.loc[kk,'nrep']=kk
    d_rep.loc[kk,'#hops']= nhop
    d_rep.loc[kk,'D_nonhop (mean)']=dmean
    d_rep.loc[kk,'D_nonhop (SE)']=std_err
    d_rep.loc[kk,'D_hop']=D2
    d_rep.loc[kk,'D_tot']=D 
    print(kk, D1)
    print('nohops:',dmean, std_err)
    print('hops:',D2)
    print('Total:',D)
    print(nhop)
    #print(rx)
    #print(t)
d_rep.to_csv(f'{path}newD_{cat}_oh{oh}_50ps.csv', index=False)

1 [1.00855897]
nohops: 1.008558971332268 0.0
hops: 0.1112890477089079
Total: 1.119848019041176
4
2 [1.00855897 0.8381825 ]
nohops: 0.9233707358362426 0.06023717899655614
hops: 0.11066482817723422
Total: 1.0340355640134768
8
3 [1.00855897 0.8381825  0.68573665]
nohops: 0.8441593747245927 0.0761290638826328
hops: 0.11881221024834926
Total: 0.9629715849729419
15
4 [1.00855897 0.8381825  0.68573665 0.58563273]
nohops: 0.7795277142143752 0.0799561316412436
hops: 0.11890652013881371
Total: 0.8984342343531889
21


In [6]:
len(t_hop)+nrep

21